<a href="https://colab.research.google.com/github/shraddhagreddy/100LinesOfCode/blob/master/CrossDataset/bc5cdr_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification

def get_spans(tags):
    """Entity spans (start, end) from BIO tags. A stray I- starts a new entity, like seqeval's default."""
    spans, start = set(), None
    for i, t in enumerate(list(tags) + ["O"]):
        if t == "O" or t.startswith("B-") or (t.startswith("I-") and start is None):
            if start is not None:
                spans.add((start, i - 1))
                start = None
            if t != "O":
                start = i
    return spans

def entity_prf(gold_seqs, pred_seqs):
    tp = fp = fn = 0
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = get_spans(g), get_spans(p)
        tp += len(G & P); fp += len(P - G); fn += len(G - P)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"P": prec, "R": rec, "F1": f1, "TP": tp, "FP": fp, "FN": fn}

# quick sanity check
gold = [["B-Disease","I-Disease","O","B-Disease"]]
pred = [["B-Disease","I-Disease","O","O"]]
print(entity_prf(gold, pred))   # expect P=1.0, R=0.5, F1≈0.667
print("GPU:", torch.cuda.is_available())

{'P': 1.0, 'R': 0.5, 'F1': 0.6666666666666666, 'TP': 1, 'FP': 0, 'FN': 1}
GPU: True


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA = "/content/test.tsv"
import os; print("file found:", os.path.exists(DATA))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
file found: True


In [4]:
def read_tsv(path):
    docs, cur = [], []
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        if not line.strip():
            if cur: docs.append(cur); cur = []
            continue
        w, t = line.split("\t")
        cur.append((w, t))
    if cur: docs.append(cur)
    return docs

def split_sentences(docs):
    sents = []
    for doc in docs:
        cur = []
        for w, t in doc:
            cur.append((w, t))
            if w == "." and t == "O":
                sents.append(cur); cur = []
        if cur: sents.append(cur)
    return sents

test_docs  = read_tsv(DATA)
test_sents = split_sentences(test_docs)
bc5_tokens = [[w for w, _ in s] for s in test_sents]
bc5_tags   = [[t for _, t in s] for s in test_sents]

print(len(test_docs), "abstracts ->", len(test_sents), "sentences")
print("entities:", sum(t == "B-Disease" for s in bc5_tags for t in s))   # should be 4420
print("longest sentence (words):", max(map(len, bc5_tokens)))
print("gold entities via get_spans:", sum(len(get_spans(t)) for t in bc5_tags))  # should also be 4420

500 abstracts -> 4988 sentences
entities: 4420
longest sentence (words): 125
gold entities via get_spans: 4420


In [5]:
!find /content/drive/MyDrive \( -name "*.safetensors" -o -name "pytorch_model.bin" \) 2>/dev/null | head
!ls /content

/content/drive/MyDrive/9444_ckpts/biobert_ncbi_seed42/model.safetensors
/content/drive/MyDrive/9444_ckpts/pubmedbert_ncbi_seed42/model.safetensors
drive  sample_data  test.tsv


In [6]:
from transformers import BertTokenizerFast, BertForTokenClassification
from google.colab import drive
drive.mount('/content/drive')

CK = "/content/drive/MyDrive/9444_ckpts"
MODELS = {
    "BioBERT":    f"{CK}/biobert_ncbi_seed42",
    "PubMedBERT": f"{CK}/pubmedbert_ncbi_seed42",
    # "BERT":     f"{CK}/bert_ncbi_seed42",   # add once Jessica shares it
}
for name, path in MODELS.items():
    tok = BertTokenizerFast.from_pretrained(path)
    m = BertForTokenClassification.from_pretrained(path)
    print(name, m.config.id2label, "| BRCA1 ->", tok.tokenize("BRCA1"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BioBERT {0: 'O', 1: 'B-Disease', 2: 'I-Disease'} | BRCA1 -> ['BR', '##CA', '##1']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

PubMedBERT {0: 'O', 1: 'B-Disease', 2: 'I-Disease'} | BRCA1 -> ['brca1']


In [7]:
import torch
device = "cuda"

def predict(model_dir, sentences, batch_size=32, max_len=256):   # 256 = same as NCBI training
    tok = BertTokenizerFast.from_pretrained(model_dir)
    model = BertForTokenClassification.from_pretrained(model_dir).to(device).eval()
    id2label = model.config.id2label
    preds, truncated = [], 0
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc = tok(batch, is_split_into_words=True, truncation=True,
                  max_length=max_len, padding=True, return_tensors="pt")
        with torch.no_grad():
            ids = model(**{k: v.to(device) for k, v in enc.items()}).logits.argmax(-1).cpu().tolist()
        for b, words in enumerate(batch):
            out, seen = ["O"] * len(words), set()
            for pos, wid in enumerate(enc.word_ids(b)):
                if wid is not None and wid not in seen:
                    out[wid] = id2label[ids[b][pos]]; seen.add(wid)
            truncated += len(seen) < len(words)
            preds.append(out)
    print(model_dir.split("/")[-1], "truncated sentences:", truncated)
    return preds

all_preds = {name: predict(path, bc5_tokens) for name, path in MODELS.items()}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

biobert_ncbi_seed42 truncated sentences: 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

pubmedbert_ncbi_seed42 truncated sentences: 0


In [8]:
rows = [{"model": n, **entity_prf(bc5_tags, p)} for n, p in all_preds.items()]
bc5_results = pd.DataFrame(rows).round(4)
bc5_results

,model,P,R,F1,TP,FP,FN
0,BioBERT,0.7013,0.6566,0.6782,2902,1236,1518
1,PubMedBERT,0.7221,0.6572,0.6881,2905,1118,1515


In [9]:
import json
OUT_EVAL = "/content/drive/MyDrive/9444_ckpts/bc5cdr_eval"
os.makedirs(OUT_EVAL, exist_ok=True)
for name, preds in all_preds.items():
    pd.DataFrame({"tokens": [" ".join(t) for t in bc5_tokens],
                  "gold":   [" ".join(t) for t in bc5_tags],
                  "pred":   [" ".join(p) for p in preds]}).to_csv(f"{OUT_EVAL}/{name}_bc5cdr_predictions.csv", index=False)
bc5_results.to_csv(f"{OUT_EVAL}/bc5cdr_results.csv", index=False)

In [10]:
def error_breakdown(gold_seqs, pred_seqs):
    ov = lambda a, b: a[0] <= b[1] and b[0] <= a[1]
    c = {"correct": 0, "boundary": 0, "false_positive": 0, "missed": 0}
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = get_spans(g), get_spans(p)
        c["correct"] += len(G & P)
        for x in P - G:
            c["boundary" if any(ov(x, y) for y in G) else "false_positive"] += 1
        for y in G - P:
            if not any(ov(y, x) for x in P): c["missed"] += 1
    return c

for name, preds in all_preds.items():
    print(f"{name:11}", error_breakdown(bc5_tags, preds))

BioBERT     {'correct': 2902, 'boundary': 640, 'false_positive': 596, 'missed': 859}
PubMedBERT  {'correct': 2905, 'boundary': 691, 'false_positive': 427, 'missed': 807}


In [11]:
import requests
from collections import Counter
BASE = "https://raw.githubusercontent.com/cambridgeltl/MTL-Bioinformatics-2016/master/data/NCBI-disease-IOB"

def load_ncbi(fname):
    sents, cur = [], []
    for line in requests.get(f"{BASE}/{fname}").text.splitlines():
        if not line.strip():
            if cur: sents.append(cur); cur = []
            continue
        cur.append(tuple(line.split("\t")))
    if cur: sents.append(cur)
    return [[w for w, _ in s] for s in sents], [[t for _, t in s] for s in sents]

ncbi_tr_tok, ncbi_tr_tag = load_ncbi("train.tsv")
ncbi_te_tok, ncbi_te_tag = load_ncbi("test.tsv")

def mentions(toks, tags):
    return [(i, s, e, " ".join(toks[i][s:e+1]).lower())
            for i, t in enumerate(tags) for s, e in get_spans(t)]

ncbi_train_set = {m[3] for m in mentions(ncbi_tr_tok, ncbi_tr_tag)}
gold_m = mentions(bc5_tokens, bc5_tags)
seen   = [m for m in gold_m if m[3] in ncbi_train_set]
unseen = [m for m in gold_m if m[3] not in ncbi_train_set]
print(f"BC5CDR test mentions: {len(gold_m)} | seen in NCBI train: {len(seen)} ({len(seen)/len(gold_m):.1%})")

for name, preds in all_preds.items():
    found = {(i, s, e) for i, p in enumerate(preds) for s, e in get_spans(p)}
    r = lambda ms: sum((i, s, e) in found for i, s, e, _ in ms) / len(ms)
    print(f"{name:11} recall seen {r(seen):.3f} | unseen {r(unseen):.3f}")

BC5CDR test mentions: 4420 | seen in NCBI train: 495 (11.2%)
BioBERT     recall seen 0.705 | unseen 0.650
PubMedBERT  recall seen 0.687 | unseen 0.654


In [12]:
for name, preds in all_preds.items():
    found = {(i, s, e) for i, p in enumerate(preds) for s, e in get_spans(p)}
    missed = Counter(m[3] for m in gold_m if m[:3] not in found)
    print(name, missed.most_common(15), "\n")

BioBERT [('seizures', 97), ('toxicity', 58), ('pain', 43), ('seizure', 40), ('cardiotoxicity', 35), ('nephropathy', 24), ('nephrotoxicity', 23), ('anxiety', 17), ('necrosis', 17), ('depression', 16), ('bleeding', 15), ('infections', 15), ('vomiting', 14), ('neurotoxicity', 13), ('convulsions', 13)] 

PubMedBERT [('seizures', 95), ('toxicity', 57), ('pain', 42), ('seizure', 41), ('nephropathy', 30), ('catalepsy', 26), ('cardiotoxicity', 21), ('hypotension', 18), ('necrosis', 16), ('chest pain', 15), ('bleeding', 15), ('nephrotoxicity', 15), ('infections', 15), ('paralysis', 15), ('neurotoxicity', 14)] 



In [13]:
def stats(toks, tags, name):
    ms = mentions(toks, tags)
    lens = [e - s + 1 for _, s, e, _ in ms]
    return {"dataset": name, "sentences": len(toks), "mentions": len(ms),
            "mentions/sent": round(len(ms)/len(toks), 2),
            "avg len (tokens)": round(sum(lens)/len(lens), 2),
            "single-token %": round(100*sum(l == 1 for l in lens)/len(lens), 1),
            "top 10": ", ".join(m for m, _ in Counter(m[3] for m in ms).most_common(10))}

ann = pd.DataFrame([stats(ncbi_te_tok, ncbi_te_tag, "NCBI test"),
                    stats(bc5_tokens, bc5_tags, "BC5CDR test")])
ann.to_csv(f"{OUT_EVAL}/annotation_stats.csv", index=False)
ann

,dataset,sentences,mentions,mentions/sent,avg len (tokens),single-token %,top 10
0,NCBI test,940,960,1.02,2.13,44.1,"dm, a - t, apc, colorectal cancer, breast canc..."
1,BC5CDR test,4988,4420,0.89,1.56,60.6,"seizures, hypertension, hypotension, toxicity,..."


In [14]:
for name, preds in all_preds.items():
    longer = shorter = shifted = 0
    for g, p in zip(bc5_tags, preds):
        G, P = get_spans(g), get_spans(p)
        for x in P - G:
            for y in G:
                if x[0] <= y[1] and y[0] <= x[1]:
                    if x[0] <= y[0] and x[1] >= y[1]: longer += 1
                    elif x[0] >= y[0] and x[1] <= y[1]: shorter += 1
                    else: shifted += 1
                    break
    print(f"{name:11} pred longer than gold {longer} | shorter {shorter} | shifted {shifted}")

# 10 examples of over-long predictions
import random; random.seed(0)
ex = []
for i, (g, p) in enumerate(zip(bc5_tags, all_preds["PubMedBERT"])):
    for x in get_spans(p) - get_spans(g):
        for y in get_spans(g):
            if x[0] <= y[0] and x[1] >= y[1] and x != y:
                ex.append((" ".join(bc5_tokens[i][y[0]:y[1]+1]), " ".join(bc5_tokens[i][x[0]:x[1]+1])))
for gold, pred in random.sample(ex, min(10, len(ex))):
    print(f"GOLD: {gold:30} PRED: {pred}")

BioBERT     pred longer than gold 517 | shorter 123 | shifted 0
PubMedBERT  pred longer than gold 593 | shorter 98 | shifted 0
GOLD: prostate cancer                PRED: metastatic androgen-independent prostate cancer
GOLD: bleeding                       PRED: intestinal bleeding
GOLD: tubulointerstial disease       PRED: chronic tubulointerstial disease
GOLD: apnoea                         PRED: succinylcholine apnoea
GOLD: interstitial nephritis         PRED: acute interstitial nephritis
GOLD: hepatitis B                    PRED: hepatitis B virus
GOLD: leukocyturia                   PRED: sterile leukocyturia
GOLD: atherosclerosis                PRED: graft atherosclerosis
GOLD: coronary arteriosclerosis      PRED: Ischaemic coronary arteriosclerosis
GOLD: aplastic anemia                PRED: acquired aplastic anemia


Cross-dataset evaluation. The NCBI-trained models were applied to the BC5CDR Disease test set (500 abstracts, 4,988 sentences, 4,420 mentions) without retraining. Entity-level F1 fell from 0.888 to 0.678 for BioBERT and from 0.866 to 0.688 for PubMedBERT. The fall came mainly from recall (≈0.90 → 0.66), not precision (≈0.86 → 0.71). The ranking between the two models reversed, but the 0.01 gap comes from a single seed and should be read as comparable performance.

Why performance drops. Three differences between the corpora explain most of the loss. (1) Different kinds of disease. Only 11.2% of BC5CDR test mentions appear in the NCBI training set. NCBI centres on hereditary diseases and cancers, while BC5CDR, built for chemical–disease relation extraction, is dominated by drug-induced conditions and symptoms. The most frequently missed mentions (seizures, toxicity, pain, cardiotoxicity, nephrotoxicity) are labelled as diseases at most a handful of times in the whole NCBI corpus. (2) A broader definition of disease. Recall on mentions that did occur in NCBI training was only slightly higher than on unseen ones (BioBERT 0.705 vs 0.650). This suggests the models learned NCBI’s narrower notion of a disease, not just its vocabulary. (3) Different boundary conventions. NCBI mentions average 2.13 tokens (44% single-token); BC5CDR mentions average 1.56 tokens (61% single-token). 81–86% of boundary errors were predictions longer than the gold span, typically because they included a modifier BC5CDR leaves out (“acute interstitial nephritis”, “metastatic androgen-independent prostate cancer”). Under lenient (overlap) matching, BC5CDR F1 rises to 0.830 for BioBERT and 0.854 for PubMedBERT, compared with about 0.95 on NCBI. Of the roughly 20-point strict drop, about half is explained by differing boundary conventions; the remaining gap reflects mentions the models do not recognise at all, mostly symptoms and drug-induced conditions that NCBI rarely annotates.

PubMedBERT produced fewer spurious entities than BioBERT (427 vs 596 false positives) but more over-long spans (691 vs 640 boundary errors), giving it higher precision at equal recall.

In [15]:
def lenient_prf(gold_seqs, pred_seqs):
    ov = lambda a, b: a[0] <= b[1] and b[0] <= a[1]
    tp_p = tp_g = n_p = n_g = 0
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = get_spans(g), get_spans(p)
        n_g += len(G); n_p += len(P)
        tp_p += sum(any(ov(x, y) for y in G) for x in P)   # predictions that overlap a gold mention
        tp_g += sum(any(ov(y, x) for x in P) for y in G)   # gold mentions overlapped by a prediction
    P_, R_ = tp_p / n_p, tp_g / n_g
    return {"P_len": round(P_, 4), "R_len": round(R_, 4), "F1_len": round(2*P_*R_/(P_+R_), 4)}

len_rows = [{"model": n, **lenient_prf(bc5_tags, p)} for n, p in all_preds.items()]
bc5_full = bc5_results.merge(pd.DataFrame(len_rows), on="model")
bc5_full.to_csv(f"{OUT_EVAL}/bc5cdr_results.csv", index=False)
bc5_full[["model", "P", "R", "F1", "P_len", "R_len", "F1_len"]]

,model,P,R,F1,P_len,R_len,F1_len
0,BioBERT,0.7013,0.6566,0.6782,0.8560,0.8057,0.8301
1,PubMedBERT,0.7221,0.6572,0.6881,0.8939,0.8174,0.8539
